<a href="https://colab.research.google.com/github/KarAnalytics/code_demos/blob/main/RAG_first_principles.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Code courtesy: This code has been adapted from towardsai.net/book code materials accompanying this book: [Building LLMs for production](https://www.oreilly.com/library/view/building-llms-for/9798324731472/)


In [1]:
!pip install -qU git+https://github.com/KarAnalytics/llm_cascade.git openai google-genai scikit-learn


  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 760.6/760.6 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 70.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.7/240.7 kB 13.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.47.0, but you have google-auth 2.49.1 which is incompatible.


In [2]:
import csv
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from sklearn.metrics.pairwise import cosine_similarity
from llm_cascade import get_cascade

llm = get_cascade()  # auto-detects API keys from env vars, .env, or Colab Secrets


LLM Cascade - available providers:
  + Gemini           model=gemini-2.5-flash
  + Ollama           model=kimi-k2.5:cloud
  + Groq             model=llama-3.3-70b-versatile
  + HuggingFace      model=meta-llama/Llama-3.3-70B-Instruct
  + Cohere           model=command-r-plus
  + OpenRouter       model=meta-llama/llama-3.3-70b-instruct:free
  + OpenAI           model=gpt-4o-mini
Not configured (skipped):
  - Grok (xAI)       (set XAI_API_KEY)


In [3]:
# False: Generate the embedding for the dataset (uses local model, no API cost).
# True: Load the dataset that already has the embedding vectors.
load_embedding = False


# Load Dataset


## Download Dataset (JSON)


The dataset includes several articles from the TowardsAI blog, which provide an in-depth explanation of the LLaMA2 model.


In [4]:
!wget https://raw.githubusercontent.com/AlaFalaki/tutorial_notebooks/main/data/mini-llama-articles.csv
!wget https://raw.githubusercontent.com/AlaFalaki/tutorial_notebooks/main/data/mini-llama-articles-with_embeddings.csv

--2026-04-08 20:38:43--  https://raw.githubusercontent.com/AlaFalaki/tutorial_notebooks/main/data/mini-llama-articles.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.109.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 173646 (170K) [text/plain]
Saving to: ‘mini-llama-articles.csv’

mini-llama-articles 100%[===================>] 169.58K  --.-KB/s    in 0.03s   

2026-04-08 20:38:44 (6.42 MB/s) - ‘mini-llama-articles.csv’ saved [173646/173646]

--2026-04-08 20:38:44--  https://raw.githubusercontent.com/AlaFalaki/tutorial_notebooks/main/data/mini-llama-articles-with_embeddings.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP r

## Read File


In [5]:
# Split the input text into chunks of specified size.
def split_into_chunks(text, chunk_size=1024):
    chunks = []
    for i in range(0, len(text), chunk_size):
        chunks.append(text[i : i + chunk_size])

    return chunks

In [6]:
chunks = []

# Load the file as a CSV
with open("./mini-llama-articles.csv", mode="r", encoding="utf-8") as file:
    csv_reader = csv.reader(file)

    for idx, row in enumerate(csv_reader):
        if idx == 0:
            continue
            # Skip header row
        chunks.extend(split_into_chunks(row[1]))

In [7]:
# Convert the JSON list to a Pandas Dataframe
df = pd.DataFrame(chunks, columns=["chunk"])

df.keys()

Index(['chunk'], dtype='object')

# Generate Embedding


In [8]:
# Embeddings use a local HuggingFace model (all-MiniLM-L6-v2)
# No API key needed — runs locally via llm.get_embedding()


In [9]:
# Generate embedding
if not load_embedding:
    print("Generating embeddings...")
    embeddings = []
    for index, row in tqdm(df.iterrows()):
        # df.at[index, 'embedding'] = llm.get_embedding( row['chunk'] )
        embeddings.append(llm.get_embedding(row["chunk"]))

    embeddings_values = pd.Series(embeddings)
    df.insert(loc=1, column="embedding", value=embeddings_values)

# Or, load the embedding from the file.
else:
    print("Loaded the embedding file.")
    # Load the file as a CSV
    df = pd.read_csv("mini-llama-articles-with_embeddings.csv")
    # Convert embedding column to an array
    df["embedding"] = df["embedding"].apply(lambda x: np.array(eval(x)), 0)

Generating embeddings...


0it [00:00, ?it/s]

  Loading embedding model: sentence-transformers/all-MiniLM-L6-v2...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

  Embedding model ready (local, no API key needed)


In [10]:
# df.to_csv('mini-llama-articles-with_embeddings.csv')

# User Question


In [11]:
# Define the user question, and convert it to embedding.
QUESTION = "How many parameters LLaMA2 model has?"
QUESTION_emb = llm.get_embedding(QUESTION)

len(QUESTION_emb)

384

# Test Cosine Similarity


Calculating the similarity of embedding representations can help us to find pieces of text that are close to each other. In the following sample you see how the Cosine Similarity metric can identify which sentence could be a possible answer for the given user question. Obviously, the unrelated answer will score lower.


In [12]:
BAD_SOURCE_emb = llm.get_embedding("The sky is blue.")
GOOD_SOURCE_emb = llm.get_embedding("LLaMA2 model has a total of 2B parameters.")

In [13]:
# A sample that how a good piece of text can achieve high similarity score compared
# to a completely unrelated text.
print("> Bad Response Score:", cosine_similarity([QUESTION_emb], [BAD_SOURCE_emb]))
print("> Good Response Score:", cosine_similarity([QUESTION_emb], [GOOD_SOURCE_emb]))

> Bad Response Score: [[-0.00069588]]
> Good Response Score: [[0.89687362]]


# Calculate Cosine Similarities


In [14]:
# The similarity between the questions and each part of the essay.
cosine_similarities = cosine_similarity([QUESTION_emb], df["embedding"].tolist())

print(cosine_similarities)

[[ 0.42389634  0.44209605  0.12147939  0.14173409  0.11352627  0.21872495
   0.25961017  0.27245391  0.34823921  0.03551458  0.14427558  0.07179068
   0.10721677  0.02094128  0.01532673  0.13963587  0.07053577  0.19606388
  -0.03592446  0.01638671  0.07672164  0.11058027  0.23705028  0.16647029
   0.06647994  0.13366314  0.10467799  0.20674347  0.19577842  0.17178742
   0.25145862  0.17147882  0.33897551  0.23842652  0.3226393   0.21220794
   0.16791321  0.19401505  0.19423016  0.15374489  0.28891772  0.20774785
   0.36797544  0.31748919  0.29309982  0.27227199  0.47771628  0.10803335
   0.24205889  0.22128891  0.37614597  0.42233826  0.63339555  0.1655476
   0.1424823   0.19828788  0.07777971  0.13091297 -0.00677211  0.11688462
   0.03082123  0.03192378  0.02501878  0.13088405  0.02129423  0.23265815
   0.18579086  0.15121353  0.29141217  0.33963518  0.17353619  0.15970106
   0.33398506  0.20751044  0.46063548  0.37146725  0.28531092  0.31389787
   0.35526827  0.2453079   0.15821577  

In [15]:
number_of_chunks_to_retrieve = 3

# Sort the scores
highest_index = np.argmax(cosine_similarities)

# Pick the N highest scored chunks
indices = np.argsort(cosine_similarities[0])[::-1][:number_of_chunks_to_retrieve]
print(indices)

[ 52  89 114]


In [16]:
# Look at the highest scored retrieved pieces of text
for idx, item in enumerate(df.chunk[indices]):
    print(f"> Chunk {idx+1}")
    print(item)
    print("----")

> Chunk 1
ma 2 by Philipp Schmid. [9] Fine-Tune Your Own Llama 2 Model in a Colab Notebook by Maxime Labonne [10]. My Github Repository
----
> Chunk 2
I. Llama 2: Revolutionizing Commercial Use Unlike its predecessor Llama 1, which was limited to research use, Llama 2 represents a major advancement as an open-source commercial model. Businesses can now integrate Llama 2 into products to create AI-powered applications. Availability on Azure and AWS facilitates fine-tuning and adoption. However, restrictions apply to prevent exploitation. Companies with over 700 million active daily users cannot use Llama 2. Additionally, its output cannot be used to improve other language models.  II. Llama 2 Model Flavors Llama 2 is available in four different model sizes: 7 billion, 13 billion, 34 billion, and 70 billion parameters. While 7B, 13B, and 70B have already been released, the 34B model is still awaited. The pretrained variant, trained on a whopping 2 trillion tokens, boasts a context window

## LLM Response Function (via llm_cascade)

Instead of a hardcoded Gemini or OpenAI client, we use `llm.generate()` from `llm_cascade`. It automatically picks the first available provider and falls back if quota is exhausted. Each call prints which provider/model was used.


In [17]:
# Using llm_cascade instead of hardcoded Gemini client

def gemini_response(system_prompt, prompt):
    """Generate a response using llm_cascade (auto-fallback across providers)."""
    try:
        response = llm.generate(prompt, system_prompt=system_prompt)
        return response.text
    except Exception as e:
        return f'Error: {e}'


# Augment the Prompt


In [18]:
from google import genai
from google.genai import types

# Use the Gemini API to answer the questions based on the retrieved pieces of text.
try:
    # Formulating the system prompt and condition the model to answer only AI-related questions.
    system_prompt = """You are an assistant and expert in answering questions from a chunks of content.
                      Only answer AI-related question, else say that you cannot answer this question."""

    # User prompt with the user's question
    prompt = """
        Read the following informations that might contain the context you require to answer the question.
        You can use the informations starting from the <START_OF_CONTEXT> tag and end with the <END_OF_CONTEXT> tag.
        Here is the content:\n\n<START_OF_CONTEXT>\n{}\n<END_OF_CONTEXT>\n\n"
        Please provide an informative and accurate answer to the following question based on the avaiable context.
        Be concise and take your time. \nQuestion: {}\nAnswer:"
      """

    # Add the retrieved pieces of text to the prompt.
    prompt = prompt.format("".join(df.chunk[indices]), QUESTION)


    response = gemini_response(system_prompt,prompt)
    print(response)

except Exception as e:
    print(f"An error occurred: {e}")


  [Gemini] unavailable, trying next...
  [Response from Ollama / kimi-k2.5:cloud]
Based on the context, LLaMA 2 is available in **four different model sizes**: **7 billion, 13 billion, 34 billion, and 70 billion parameters**. Note that while the 7B, 13B, and 70B models have been released, the 34B model was still awaited at the time of the writing.


## How Augmenting the Prompt can address knowledge cutoff limitations and hallucinations

In [19]:
# Consider this as a retrieved chunk
# https://ai.meta.com/blog/llama-4-multimodal-intelligence/

Example_chunk = """
# Llama 4 Technical Overview

Meta has introduced the Llama 4 model family, marking a significant advancement in open-weight multimodal AI with three distinct variants designed for different use cases. The Llama 4 Scout represents the more compact option with 17 billion active parameters utilizing 16 experts within a total parameter count of 109 billion, designed to fit on a single H100 GPU with Int4 quantization. The Llama 4 Maverick serves as the flagship model with 17 billion active parameters but employs 128 experts across 400 billion total parameters, fitting on a single H100 host and designed as the primary workhorse for general assistant and chat applications. The preview model Llama 4 Behemoth functions as a teacher model with 288 billion active parameters, 16 experts, and nearly 2 trillion total parameters, currently still in training but demonstrating state-of-the-art performance on mathematical and reasoning benchmarks.
The architecture represents Meta's first implementation of mixture-of-experts (MoE) design in the Llama series, fundamentally changing how the models process information by activating only a fraction of total parameters for each token. The MoE architecture alternates between dense and mixture-of-experts layers, with the Maverick model specifically using 128 routed experts alongside a shared expert, where each token is processed by the shared expert and routed to one of the 128 specialized experts. This design significantly improves compute efficiency during both training and inference while delivering higher quality outputs compared to dense models with equivalent computational budgets.
Native multimodality represents another breakthrough, achieved through early fusion architecture that seamlessly integrates text and vision tokens into a unified model backbone during pre-training. This approach enables joint training with massive amounts of unlabeled text, image, and video data, with the vision encoder based on MetaCLIP but trained separately in conjunction with a frozen Llama model to optimize adaptation to the language model. The models support up to 48 images during pre-training and demonstrate strong performance with up to 8 images in post-training scenarios, enabling sophisticated visual reasoning and understanding tasks across multiple input modalities.
The training infrastructure incorporates several technical innovations, including the MetaP technique for reliable hyperparameter optimization that transfers well across different model configurations and training scenarios. The models were trained on over 30 trillion tokens, representing more than double the training data of Llama 3, with support for 200 languages including over 100 languages with more than 1 billion tokens each. Training efficiency was maximized through FP8 precision without quality sacrifice, achieving 390 TFLOPs per GPU utilization on 32,000 GPUs during Behemoth pre-training, while a specialized mid-training phase enhanced core capabilities and enabled the industry-leading 10 million token context length for Scout.
Context length capabilities represent a dramatic advancement, with Llama 4 Scout supporting 10 million tokens compared to Llama 3's 128K limit through the innovative iRoPE architecture that employs interleaved attention layers without positional embeddings and inference-time temperature scaling of attention. This architecture enables superior length generalization and opens possibilities for multi-document summarization, extensive user activity parsing for personalization, and reasoning over vast codebases, with compelling performance demonstrated in retrieval tasks and cumulative negative log-likelihood evaluations over 10 million tokens of code.
The post-training pipeline underwent significant revision with a three-stage approach consisting of lightweight supervised fine-tuning, online reinforcement learning, and lightweight direct preference optimization. A critical insight involved removing over 50% of data tagged as easy using Llama models as judges, focusing training on harder examples to prevent over-constraining that could restrict exploration during online RL and lead to suboptimal performance in reasoning, coding, and mathematics. The continuous online RL strategy alternated between model training and adaptive data filtering to retain only medium-to-hard difficulty prompts, proving highly beneficial for compute and accuracy trade-offs.
Safety and bias mitigation received comprehensive attention through multi-layered approaches spanning pre-training data filtering, post-training policy conformance, and system-level safeguards. Meta introduced Generative Offensive Agent Testing (GOAT) to address traditional red-teaming limitations by simulating multi-turn interactions of medium-skilled adversarial actors, enabling more efficient vulnerability detection while allowing human experts to focus on novel adversarial areas. Particular emphasis was placed on addressing political and social bias inherent in internet training data, with Llama 4 demonstrating significant improvements over Llama 3 in presenting balanced perspectives on contentious issues without favoring particular viewpoints.
The distillation process for creating smaller models from Behemoth required novel approaches, including a dynamic loss function that weighted soft and hard targets throughout training, with codistillation during pre-training amortizing computational costs for the majority of training data. Post-training the 2 trillion parameter Behemoth model necessitated pruning 95% of supervised fine-tuning data compared to 50% for smaller models, followed by large-scale reinforcement learning focused on sampling hard prompts through pass@k analysis and constructing training curricula of increasing difficulty. The unprecedented scale required complete infrastructure overhaul, including optimized MoE parallelization and fully asynchronous online RL training framework that improved training efficiency by approximately 10x over previous generations through flexible GPU allocation and resource balancing across multiple models.
"""

In [20]:
QUESTION = "How many parameters LLaMA 4 model has?"

# Formulating the system prompt
system_prompt = """ You are an assistant and expert in answering questions from a chunks of content.
                    Only answer AI-related question, else say that you cannot answer this question."""

# Combining the system prompt with the user's question
prompt = """Read the following informations that might contain the context you require to answer the question.
            You can use the informations starting from the <START_OF_CONTEXT> tag and end with the <END_OF_CONTEXT> tag.
            Here is the content:\n\n<START_OF_CONTEXT>\n{}\n<END_OF_CONTEXT>\n\n".
            Please provide an informative and accurate answer to the following question based on the avaiable context.
            Be concise and take your time. \nQuestion: {}\nAnswer:
            """

prompt = prompt.format(Example_chunk, QUESTION)

response = gemini_response(system_prompt,prompt)
print(response)

  [Gemini] unavailable, trying next...
  [Response from Ollama / kimi-k2.5:cloud]
Based on the context provided, the Llama 4 model family consists of three variants with different parameter sizes:

**Llama 4 Scout:** 109 billion total parameters (17 billion active parameters)

**Llama 4 Maverick:** 400 billion total parameters (17 billion active parameters) 

**Llama 4 Behemoth:** Nearly 2 trillion total parameters (288 billion active parameters)

Note that these models use a Mixture-of-Experts (MoE) architecture, where only a fraction of the total parameters are activated per token (the "active parameters"), while the "total parameters" refers to the complete model size.


# Without Augmentation


In [21]:
QUESTION = "How many parameters LLaMA 4 model has?"

# Formulating the system prompt
system_prompt = "You are an assistant and expert in answering questions."

# Combining the system prompt with the user's question
prompt = "Be concise and take your time to answer the following question. \nQuestion: {}\nAnswer:"

prompt = prompt.format(QUESTION)

response = gemini_response(system_prompt,prompt)
print(response)

  [Gemini] unavailable, trying next...
  [Response from Ollama / kimi-k2.5:cloud]
LLaMA 4 has not been released or announced by Meta. As of early 2024, the latest version is **LLaMA 3**, which comes in three sizes: **8B**, **70B**, and **405B** parameters.
